# EfficientNet Image Retrieval Evaluation
Évaluation des modèles **EfficientNetB0** et **EfficientNetB4** pré-entraînés sur ImageNet pour la recherche d'images par similarité visuelle.

**Métriques calculées :**
- Recall@K (K = 1, 5, 10, 20)
- Precision@K (K = 1, 5, 10, 20)
- Courbe Precision-Recall
- AP (Average Precision) par requête
- mAP (mean Average Precision)
- Accuracy@1 (Top-1)

## 0. Google Drive & Installation

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q timm faiss-cpu scikit-learn matplotlib pandas tqdm Pillow

## 1. Configuration
**Modifiez les chemins selon votre structure Drive.**

In [ ]:
import os

# ─── À MODIFIER ────────────────────────────────────────────────────────────────
DRIVE_BASE   = "/content/drive/MyDrive"
# Dossier contenant les images (ex: BMW_Serie3Berline_1.jpg ou sous-dossiers par classe)
DATASET_DIR  = os.path.join(DRIVE_BASE, "MIR_Project", "dataset")
# Dossier où les résultats seront sauvegardés
RESULTS_DIR  = os.path.join(DRIVE_BASE, "MIR_Project", "results", "efficientnet")
# ───────────────────────────────────────────────────────────────────────────────

# Format du nom de fichier pour extraire la classe
# Exemples supportés :
#   Sous-dossiers  : dataset/BMW_Serie3/img1.jpg  -> label = nom du dossier
#   Nom de fichier : 0_0_BMW_Serie3_1.jpg         -> label = token n°2 (séparateur _)
LABEL_FROM_FOLDER = False   # True = sous-dossiers, False = parsing du nom de fichier
FILENAME_CLASS_TOKEN = 2    # index du token dans le nom (ex: "0_0_BMW_Serie3_1" -> token[2] = "BMW")
# Si LABEL_FROM_FOLDER=False, la classe est reconstruite comme "_".join(tokens[FILENAME_CLASS_TOKEN:-1])
# ex: '0_0_BMW_Serie3Berline_1.jpg' -> tokens = ['0','0','BMW','Serie3Berline','1'] -> 'BMW_Serie3Berline'

BATCH_SIZE   = 64
IMAGE_SIZE   = 224   # 380 pour B4 en résolution native, 224 accepté aussi
NUM_WORKERS  = 2
K_VALUES     = [1, 5, 10, 20]   # Recall/Precision@K

os.makedirs(RESULTS_DIR, exist_ok=True)
print(f"Dataset : {DATASET_DIR}")
print(f"Résultats : {RESULTS_DIR}")

## 2. Imports

In [ ]:
import json
import time
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

import timm
import faiss
from sklearn.metrics import precision_recall_curve, average_precision_score

warnings.filterwarnings('ignore')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device : {DEVICE}")
if DEVICE == 'cuda':
    print(f"GPU : {torch.cuda.get_device_name(0)}")

## 3. Chargement du Dataset

In [ ]:
VALID_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

def extract_label_from_filename(path: Path) -> str:
    """0_0_BMW_Serie3Berline_1.jpg → 'BMW_Serie3Berline'"""
    stem = path.stem  # sans extension
    tokens = stem.split('_')
    if len(tokens) <= FILENAME_CLASS_TOKEN + 1:
        return stem
    # tokens[FILENAME_CLASS_TOKEN:-1] = classe (multi-token), dernier token = index image
    return '_'.join(tokens[FILENAME_CLASS_TOKEN:-1])


class ImageDataset(Dataset):
    def __init__(self, root: str, transform=None, label_from_folder=False):
        self.root = Path(root)
        self.transform = transform
        self.samples = []   # list of (path, label_str)
        self.labels  = []   # list of int
        self.label2idx = {}

        if label_from_folder:
            for cls_dir in sorted(self.root.iterdir()):
                if not cls_dir.is_dir():
                    continue
                for img_path in sorted(cls_dir.iterdir()):
                    if img_path.suffix.lower() in VALID_EXTS:
                        self.samples.append((img_path, cls_dir.name))
        else:
            for img_path in sorted(self.root.iterdir()):
                if img_path.suffix.lower() in VALID_EXTS:
                    label = extract_label_from_filename(img_path)
                    self.samples.append((img_path, label))

        # Encode labels
        unique_labels = sorted(set(l for _, l in self.samples))
        self.label2idx = {l: i for i, l in enumerate(unique_labels)}
        self.idx2label = {i: l for l, i in self.label2idx.items()}
        self.labels = [self.label2idx[l] for _, l in self.samples]

        print(f"  {len(self.samples)} images | {len(unique_labels)} classes")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, _ = self.samples[idx]
        img = Image.open(path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, self.labels[idx]


def get_transform(img_size=224):
    return transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225]),
    ])


print("Chargement du dataset...")
dataset = ImageDataset(DATASET_DIR, transform=get_transform(IMAGE_SIZE),
                       label_from_folder=LABEL_FROM_FOLDER)
loader  = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False,
                     num_workers=NUM_WORKERS, pin_memory=(DEVICE == 'cuda'))

labels_np = np.array(dataset.labels)
print(f"Labels shape : {labels_np.shape}")
# Distribution des classes
unique, counts = np.unique(labels_np, return_counts=True)
print(f"Min images/classe : {counts.min()} | Max : {counts.max()} | Moyenne : {counts.mean():.1f}")

## 4. Extraction des Features

In [ ]:
def build_feature_extractor(model_name: str) -> nn.Module:
    """
    Charge un EfficientNet pré-entraîné (timm) sans la tête de classification.
    model_name : 'efficientnet_b0' ou 'efficientnet_b4'
    """
    model = timm.create_model(model_name, pretrained=True, num_classes=0)  # num_classes=0 -> pas de fc
    model = model.to(DEVICE).eval()
    # Taille de l'embedding
    dummy = torch.zeros(1, 3, IMAGE_SIZE, IMAGE_SIZE).to(DEVICE)
    with torch.no_grad():
        feat_dim = model(dummy).shape[-1]
    print(f"  {model_name} | embedding dim = {feat_dim}")
    return model, feat_dim


@torch.no_grad()
def extract_features(model: nn.Module, loader: DataLoader) -> np.ndarray:
    all_feats = []
    for imgs, _ in tqdm(loader, desc="  Extraction", leave=False):
        imgs = imgs.to(DEVICE)
        feats = model(imgs)           # (B, D)
        feats = torch.nn.functional.normalize(feats, dim=1)  # L2-norm
        all_feats.append(feats.cpu().numpy())
    return np.vstack(all_feats).astype(np.float32)

## 5. Métriques de Retrieval

In [ ]:
def build_faiss_index(embeddings: np.ndarray) -> faiss.IndexFlatIP:
    """Index Inner Product (= cosine sim sur vecteurs normalisés L2)"""
    dim = embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(embeddings)
    return index


def compute_retrieval_metrics(embeddings: np.ndarray,
                               labels: np.ndarray,
                               k_values: list,
                               n_queries: int = None) -> dict:
    """
    Calcule Recall@K, Precision@K, AP et mAP pour chaque requête.
    - embeddings : (N, D) float32, L2-normalisés
    - labels     : (N,) int
    - k_values   : liste de K pour Recall@K et Precision@K
    - n_queries  : si None, toutes les images sont requêtes
    """
    N = len(embeddings)
    index = build_faiss_index(embeddings)

    max_k = max(k_values)
    # +1 car la requête elle-même est dans la base
    _, I = index.search(embeddings, max_k + 1)

    query_ids = list(range(N)) if n_queries is None else list(range(n_queries))

    recall_at_k    = {k: [] for k in k_values}
    precision_at_k = {k: [] for k in k_values}
    ap_list        = []

    for q_idx in query_ids:
        q_label = labels[q_idx]
        # Nombre total de positifs dans la base (excl. la requête elle-même)
        n_relevant = int((labels == q_label).sum()) - 1
        if n_relevant == 0:
            continue

        # Résultats (on exclut la requête elle-même)
        retrieved = [i for i in I[q_idx] if i != q_idx][:max_k]
        relevance = (labels[retrieved] == q_label).astype(int)   # 1 si pertinent

        # ── Recall@K et Precision@K ───────────────────────────────────────────
        for k in k_values:
            top_k_rel = relevance[:k].sum()
            recall_at_k[k].append(top_k_rel / n_relevant)
            precision_at_k[k].append(top_k_rel / k)

        # ── AP (Average Precision) ────────────────────────────────────────────
        # AP = somme(Precision@i * delta_Recall@i) pour tous les rang i
        n_ret = len(relevance)
        cumsum = np.cumsum(relevance)
        precision_at_i = cumsum / (np.arange(n_ret) + 1)
        ap = float((precision_at_i * relevance).sum() / n_relevant)
        ap_list.append(ap)

    results = {
        'mAP': float(np.mean(ap_list)),
        'AP_per_query': ap_list,
    }
    for k in k_values:
        results[f'Recall@{k}']    = float(np.mean(recall_at_k[k]))
        results[f'Precision@{k}'] = float(np.mean(precision_at_k[k]))
    results['Accuracy@1'] = results['Precision@1']   # alias
    return results


def compute_pr_curve(embeddings: np.ndarray, labels: np.ndarray,
                     n_queries: int = 200) -> dict:
    """
    Calcule la courbe Precision-Recall moyenne sur n_queries requêtes.
    Retourne interpolation sur 100 points de recall communs.
    """
    N = len(embeddings)
    index = build_faiss_index(embeddings)
    # Chercher tous les voisins pour construire la courbe complète
    _, I = index.search(embeddings[:n_queries], N)

    recall_grid = np.linspace(0, 1, 101)
    precision_interp_all = []

    for q_idx in range(n_queries):
        q_label = labels[q_idx]
        retrieved = [i for i in I[q_idx] if i != q_idx]
        if not retrieved:
            continue
        relevance = (labels[retrieved] == q_label).astype(int)
        n_relevant = relevance.sum()
        if n_relevant == 0:
            continue

        scores = np.linspace(1, 0, len(relevance))  # rang croissant -> score décroissant
        prec, rec, _ = precision_recall_curve(relevance, scores)
        # Interpoler sur la grille commune
        prec_interp = np.interp(recall_grid, rec[::-1], prec[::-1])
        precision_interp_all.append(prec_interp)

    mean_precision = np.mean(precision_interp_all, axis=0)
    return {'recall': recall_grid.tolist(), 'precision': mean_precision.tolist()}


print("Fonctions de métriques définies.")

## 6. Évaluation des Modèles

In [ ]:
MODELS_TO_TEST = [
    ('efficientnet_b0', 224),
    ('efficientnet_b4', 380),   # résolution native B4
]

all_results   = {}   # model_name -> metrics dict
all_pr_curves = {}   # model_name -> {recall, precision}
all_embeddings = {}  # model_name -> np.ndarray  (gardé pour visualisation)

for model_name, img_size in MODELS_TO_TEST:
    print(f"\n{'='*60}")
    print(f"Modèle : {model_name}  |  Image size : {img_size}")
    print('='*60)

    # Reconstruire le loader à la bonne résolution
    ds = ImageDataset(DATASET_DIR, transform=get_transform(img_size),
                      label_from_folder=LABEL_FROM_FOLDER)
    dl = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False,
                    num_workers=NUM_WORKERS, pin_memory=(DEVICE == 'cuda'))

    model, feat_dim = build_feature_extractor(model_name)

    t0 = time.time()
    embeddings = extract_features(model, dl)
    t_feat = time.time() - t0
    print(f"  Features extraites en {t_feat:.1f}s | shape : {embeddings.shape}")

    # Métriques de retrieval
    print("  Calcul des métriques...")
    metrics = compute_retrieval_metrics(embeddings, labels_np, K_VALUES)
    metrics['model']          = model_name
    metrics['image_size']     = img_size
    metrics['feat_dim']       = feat_dim
    metrics['extraction_time_s'] = round(t_feat, 2)
    metrics['n_images']       = len(embeddings)
    metrics['n_classes']      = len(dataset.label2idx)
    metrics['timestamp']      = datetime.now().isoformat()

    all_results[model_name]    = metrics
    all_embeddings[model_name] = embeddings

    # Courbe P-R (sur 200 premières requêtes max)
    n_q = min(200, len(embeddings))
    print(f"  Calcul courbe P-R ({n_q} requêtes)...")
    all_pr_curves[model_name] = compute_pr_curve(embeddings, labels_np, n_queries=n_q)

    print(f"  mAP          : {metrics['mAP']:.4f}")
    print(f"  Recall@10    : {metrics['Recall@10']:.4f}")
    print(f"  Precision@10 : {metrics['Precision@10']:.4f}")
    print(f"  Accuracy@1   : {metrics['Accuracy@1']:.4f}")

    # Libérer la mémoire GPU
    del model
    torch.cuda.empty_cache() if DEVICE == 'cuda' else None

print("\nÉvaluation terminée.")

## 7. Sauvegarde des Résultats

In [ ]:
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

# ── JSON complet ─────────────────────────────────────────────────────────────
json_path = os.path.join(RESULTS_DIR, f"efficientnet_results_{timestamp}.json")
save_data = {
    'results': {
        k: {key: val for key, val in v.items() if key != 'AP_per_query'}
        for k, v in all_results.items()
    },
    'pr_curves': all_pr_curves,
    'config': {
        'dataset_dir': DATASET_DIR,
        'image_size': IMAGE_SIZE,
        'batch_size': BATCH_SIZE,
        'k_values': K_VALUES,
        'device': DEVICE,
    }
}
with open(json_path, 'w') as f:
    json.dump(save_data, f, indent=2)
print(f"JSON sauvegardé : {json_path}")

# ── CSV résumé ───────────────────────────────────────────────────────────────
rows = []
for model_name, m in all_results.items():
    row = {'model': model_name}
    for k in K_VALUES:
        row[f'Recall@{k}']    = round(m[f'Recall@{k}'], 4)
        row[f'Precision@{k}'] = round(m[f'Precision@{k}'], 4)
    row['mAP']         = round(m['mAP'], 4)
    row['Accuracy@1']  = round(m['Accuracy@1'], 4)
    row['feat_dim']    = m['feat_dim']
    row['image_size']  = m['image_size']
    row['time_s']      = m['extraction_time_s']
    rows.append(row)

df = pd.DataFrame(rows)
csv_path = os.path.join(RESULTS_DIR, f"efficientnet_summary_{timestamp}.csv")
df.to_csv(csv_path, index=False)
print(f"CSV sauvegardé  : {csv_path}")

# ── Embeddings numpy ─────────────────────────────────────────────────────────
for model_name, emb in all_embeddings.items():
    npy_path = os.path.join(RESULTS_DIR, f"embeddings_{model_name}_{timestamp}.npy")
    np.save(npy_path, emb)
    print(f"Embeddings sauvegardés : {npy_path}")

# Affichage tableau
print("\n" + "="*80)
print(df.to_string(index=False))
print("="*80)

## 8. Visualisations

In [ ]:
# ── Courbes Precision-Recall ──────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))
colors = ['#2196F3', '#F44336', '#4CAF50', '#FF9800']

for (model_name, curve), color in zip(all_pr_curves.items(), colors):
    m = all_results[model_name]
    label = f"{model_name} (mAP={m['mAP']:.3f})"
    ax.plot(curve['recall'], curve['precision'], label=label, color=color, linewidth=2)

ax.set_xlabel('Recall', fontsize=12)
ax.set_ylabel('Precision', fontsize=12)
ax.set_title('Courbe Precision-Recall (moyenne sur requêtes)', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 1); ax.set_ylim(0, 1)

pr_path = os.path.join(RESULTS_DIR, f"pr_curve_{timestamp}.png")
plt.tight_layout()
plt.savefig(pr_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Courbe P-R sauvegardée : {pr_path}")

In [ ]:
# ── Barres Recall@K et Precision@K ───────────────────────────────────────────
model_names = list(all_results.keys())
n_models = len(model_names)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
x = np.arange(len(K_VALUES))
width = 0.35

for i, (model_name, color) in enumerate(zip(model_names, ['#2196F3', '#F44336'])):
    m = all_results[model_name]
    recall_vals    = [m[f'Recall@{k}']    for k in K_VALUES]
    precision_vals = [m[f'Precision@{k}'] for k in K_VALUES]
    offset = (i - (n_models - 1) / 2) * width
    axes[0].bar(x + offset, recall_vals,    width, label=model_name, color=color, alpha=0.85)
    axes[1].bar(x + offset, precision_vals, width, label=model_name, color=color, alpha=0.85)

for ax, title in zip(axes, ['Recall@K', 'Precision@K']):
    ax.set_xticks(x)
    ax.set_xticklabels([f'K={k}' for k in K_VALUES])
    ax.set_ylabel(title)
    ax.set_title(title)
    ax.legend()
    ax.set_ylim(0, 1)
    ax.grid(True, axis='y', alpha=0.3)

bar_path = os.path.join(RESULTS_DIR, f"recall_precision_bar_{timestamp}.png")
plt.suptitle('Recall@K et Precision@K — EfficientNet (pré-entraîné)', fontsize=13)
plt.tight_layout()
plt.savefig(bar_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Graphe sauvegardé : {bar_path}")

In [ ]:
# ── Comparaison mAP ──────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 4))
model_names = list(all_results.keys())
map_values  = [all_results[m]['mAP'] for m in model_names]
colors_bar  = ['#2196F3', '#F44336']

bars = ax.bar(model_names, map_values, color=colors_bar[:len(model_names)], alpha=0.85, width=0.4)
for bar, val in zip(bars, map_values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
            f'{val:.4f}', ha='center', va='bottom', fontsize=12, fontweight='bold')

ax.set_ylabel('mAP', fontsize=12)
ax.set_title('mAP — EfficientNet pré-entraîné (ImageNet)', fontsize=13)
ax.set_ylim(0, min(1.0, max(map_values) * 1.2))
ax.grid(True, axis='y', alpha=0.3)

map_path = os.path.join(RESULTS_DIR, f"mAP_comparison_{timestamp}.png")
plt.tight_layout()
plt.savefig(map_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Graphe mAP sauvegardé : {map_path}")

In [ ]:
# ── Résultats Qualitatifs : Requêtes → Top-5 ─────────────────────────────────
N_QUERIES_VIZ = 3   # nombre de requêtes à afficher
K_VIZ = 5

def show_retrieval_results(model_name: str, embeddings: np.ndarray, n_viz=3, k=5):
    index = build_faiss_index(embeddings)
    query_ids = np.random.choice(len(embeddings), n_viz, replace=False)

    fig, axes = plt.subplots(n_viz, k + 1, figsize=(2.5 * (k + 1), 2.5 * n_viz))
    if n_viz == 1:
        axes = axes[np.newaxis, :]

    _, I = index.search(embeddings[query_ids], k + 1)

    for row, q_idx in enumerate(query_ids):
        q_label = dataset.idx2label[labels_np[q_idx]]
        retrieved = [i for i in I[row] if i != q_idx][:k]

        # Requête
        img_q = Image.open(dataset.samples[q_idx][0]).convert('RGB').resize((100, 100))
        axes[row, 0].imshow(img_q)
        axes[row, 0].set_title(f"Requête\n{q_label[:15]}", fontsize=8, color='blue')
        axes[row, 0].axis('off')

        # Top-k
        for col, r_idx in enumerate(retrieved):
            r_label = dataset.idx2label[labels_np[r_idx]]
            img_r = Image.open(dataset.samples[r_idx][0]).convert('RGB').resize((100, 100))
            axes[row, col + 1].imshow(img_r)
            is_rel = (r_label == q_label)
            color = 'green' if is_rel else 'red'
            axes[row, col + 1].set_title(f"{r_label[:12]}", fontsize=7, color=color)
            axes[row, col + 1].axis('off')
            # Bordure colorée
            for spine in axes[row, col + 1].spines.values():
                spine.set_edgecolor(color); spine.set_linewidth(3); spine.set_visible(True)

    fig.suptitle(f'{model_name} — Top-{k} résultats (vert=pertinent, rouge=non-pertinent)',
                 fontsize=11)
    plt.tight_layout()
    qual_path = os.path.join(RESULTS_DIR, f"qualitative_{model_name}_{timestamp}.png")
    plt.savefig(qual_path, dpi=120, bbox_inches='tight')
    plt.show()
    print(f"Sauvegardé : {qual_path}")


for model_name in all_embeddings:
    print(f"\nVisualisations pour {model_name}...")
    show_retrieval_results(model_name, all_embeddings[model_name],
                           n_viz=N_QUERIES_VIZ, k=K_VIZ)

## 9. Résumé Final

In [ ]:
print("\n" + "═"*70)
print(" RÉSULTATS FINAUX — EfficientNet Pré-entraîné (ImageNet)")
print("═"*70)
for model_name, m in all_results.items():
    print(f"\n  {model_name}  (img_size={m['image_size']}, feat_dim={m['feat_dim']})")
    print(f"  {'─'*40}")
    print(f"  mAP          : {m['mAP']:.4f}")
    print(f"  Accuracy@1   : {m['Accuracy@1']:.4f}")
    for k in K_VALUES:
        print(f"  Recall@{k:<3}   : {m[f'Recall@{k}']:.4f}  |  Precision@{k:<3}: {m[f'Precision@{k}']:.4f}")
    print(f"  Temps extract : {m['extraction_time_s']}s")
print("═"*70)

print(f"\nFichiers sauvegardés dans : {RESULTS_DIR}")
for f in sorted(Path(RESULTS_DIR).iterdir()):
    if timestamp in f.name:
        size_kb = f.stat().st_size // 1024
        print(f"  {f.name}  ({size_kb} KB)")